In [1]:
# =============================================================================
# STEP 3 - CALIBRATION QUALITY ON CIC-IoT-2023
#
# Figure 6 and the calibration table cover three environments; the fourth was
# never measured, so the caption currently has to say so. This computes the same
# quantities for CIC-IoT-2023 on the source calibration pool, which is held out
# from the isotonic fitting partition, and regenerates the figure with all four.
#
# The purpose is unchanged: if the probabilities feeding the conformal layer are
# well calibrated in distribution, then coverage behaviour under shift cannot be
# dismissed as a base-miscalibration artefact. On this dataset that argument runs
# the other way as well, since nothing fails and a reader could otherwise wonder
# whether the null is an artefact of unusually good or bad calibration.
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config
import numpy as np, pandas as pd
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY
NBINS=15          # equal-mass bins, the nb26 convention, so rows are comparable
print('ready')


Mounted at /content/drive
ready


In [2]:
# =============================================================================
# Cell 2 - the same estimators nb26 used, so the new rows are comparable with the
# existing three datasets rather than measured on a different scale.
# =============================================================================
def brier_ovr(P,y,K):
    return {k: float(np.mean((P[:,k]-(y==k).astype(float))**2)) for k in range(K)}

def ece_equal_mass_1d(p, correct, nbins=NBINS):
    if len(p)<nbins: nbins=max(1,len(p)//2)
    o=np.argsort(p); p,correct=p[o],correct[o]
    return float(sum(len(b)/len(p)*abs(correct[b].mean()-p[b].mean())
                     for b in np.array_split(np.arange(len(p)),nbins) if len(b)))

def ece_ovr(P,y,K,nbins=NBINS):
    return {k: ece_equal_mass_1d(P[:,k],(y==k).astype(float),nbins) for k in range(K)}

def ece_toplabel(P,y,nbins=NBINS):
    return ece_equal_mass_1d(P.max(1),(P.argmax(1)==y).astype(float),nbins)

iot=pd.read_parquet(config.PROC_DIR/'ciciot2023_prepared.parquet')
sp =pd.read_parquet(config.PROC_DIR/'ciciot2023_split.parquet')
iot['side']=sp['side'].values; iot['partition']=sp['partition'].values
mrec=json.loads((RD/'ciciot2023_model_record.json').read_text())
CLASSES=mrec['classes_canonical_order']; K=len(CLASSES); c2i={c:i for i,c in enumerate(CLASSES)}
iot['y']=iot['family'].map(c2i).astype(np.int64)
sc_idx=iot.index[iot.partition=='source_cal_pool'].to_numpy()
y_sc=iot['y'].to_numpy()[sc_idx]
PROBS=config.DATA_DIR/'ciciot_probs'
files=sorted(PROBS.glob('ciciot2023__*.npz'))
assert len(files)==30, f'expected 30 probability files, found {len(files)}'
d0=np.load(files[0])
assert d0['srcpool'].shape[0]==len(sc_idx), \
    'stale probabilities: delete data/ciciot_probs/*.npz and re-run nb33'
assert list(d0['classes'].astype(str))==CLASSES, 'cached class order differs from the model record'
print('classes:', CLASSES)
print('source calibration pool:', len(sc_idx), '| class counts:',
      {CLASSES[k]:int((y_sc==k).sum()) for k in range(K)})


classes: ['Benign', 'BruteForce', 'DDoS', 'DoS', 'Mirai', 'Recon', 'Spoofing', 'Web']
source calibration pool: 158079 | class counts: {'Benign': 6299, 'BruteForce': 1371, 'DDoS': 68476, 'DoS': 25215, 'Mirai': 18953, 'Recon': 23055, 'Spoofing': 12575, 'Web': 2135}


In [3]:
# =============================================================================
# Cell 3 - measure. The source calibration pool is disjoint from D_probcal, where
# the isotonic calibrator was fitted, so this is an honest held-out check and not
# the calibrator being scored on its own fitting data.
# =============================================================================
rows=[]; top=[]
for f in files:
    _,arch,sd=f.stem.split('__'); seed=int(sd.replace('seed',''))
    P=np.load(f)['srcpool'].astype(np.float64)
    br=brier_ovr(P,y_sc,K); ec=ece_ovr(P,y_sc,K)
    for k in range(K):
        rows.append({'dataset':'ciciot2023','arch':arch,'seed':seed,'class':CLASSES[k],
                     'brier':br[k],'ece':ec[k]})
    top.append({'dataset':'ciciot2023','arch':arch,'seed':seed,'toplabel_ece':ece_toplabel(P,y_sc)})
new=pd.DataFrame(rows); tl=pd.DataFrame(top)
summ=new.groupby('class').agg(brier=('brier','mean'), ece=('ece','mean')).round(5)
print('CIC-IoT-2023 calibration on the held-out source pool:')
print(summ.to_string())
print(f'\ntop-label ECE: {tl.toplabel_ece.mean():.5f}')
print(f'max per-class ECE: {new.ece.max():.5f}  (the other three datasets are all <= 0.002)')

# append into the shared table, replacing any prior ciciot rows
base=RD/'calibration_quality.csv'
old=pd.read_csv(base)
add=new.groupby(['dataset','class'],as_index=False).agg(brier=('brier','mean'), ece=('ece','mean')).round(6)
merged=pd.concat([old[old.dataset!='ciciot2023'], add[old.columns.intersection(add.columns)]],
                 ignore_index=True)
merged.to_csv(base, index=False)
print(f'\ncalibration_quality.csv: {len(old)} -> {len(merged)} rows')
print(merged.groupby('dataset').agg(mean_ece=('ece','mean'), max_ece=('ece','max')).round(5).to_string())
new.to_csv(RD/'calibration_quality_ciciot2023_cells.csv', index=False)
tl.to_csv(RD/'calibration_toplabel_ciciot2023.csv', index=False)


CIC-IoT-2023 calibration on the held-out source pool:
              brier      ece
class                       
Benign      0.01143  0.00046
BruteForce  0.00436  0.00058
DDoS        0.00205  0.00026
DoS         0.00160  0.00021
Mirai       0.00025  0.00012
Recon       0.02126  0.00093
Spoofing    0.01663  0.00147
Web         0.00699  0.00048

top-label ECE: 0.00134
max per-class ECE: 0.00271  (the other three datasets are all <= 0.002)

calibration_quality.csv: 12 -> 20 rows
            mean_ece  max_ece
dataset                      
cicids2017   0.00010  0.00010
ciciot2023   0.00056  0.00147
nslkdd       0.00020  0.00050
ugr16        0.00118  0.00190


In [4]:
# =============================================================================
# Cell 4 - regenerate Figure 6 with all four environments.
# =============================================================================
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.spines.top':False,
                     'axes.spines.right':False,'figure.dpi':300})
C={'nslkdd':'#2f4b7c','cicids2017':'#c65911','ugr16':'#4a5a2f','ciciot2023':'#7b3f8f'}
L={'nslkdd':'NSL-KDD','cicids2017':'CIC-IDS2017','ugr16':"UGR'16",'ciciot2023':'CIC-IoT-2023'}
cq=pd.read_csv(RD/'calibration_quality.csv')
order=[d for d in ['nslkdd','cicids2017','ugr16','ciciot2023'] if d in set(cq.dataset)]
cq['ord']=cq.dataset.map({d:i for i,d in enumerate(order)})
cq=cq.sort_values(['ord','class']).reset_index(drop=True)
x=np.arange(len(cq)); cols=[C[d] for d in cq.dataset]
fig,axs=plt.subplots(1,2,figsize=(11.0,4.4))
for ax,metric,ttl in [(axs[0],'ece','(a) expected calibration error'),(axs[1],'brier','(b) Brier score')]:
    ax.bar(x,cq[metric],color=cols,edgecolor='white',width=0.74,zorder=3)
    ax.set_xticks(x); ax.set_xticklabels(cq['class'],rotation=55,ha='right',fontsize=7)
    ax.set_ylabel('ECE' if metric=='ece' else 'Brier'); ax.set_title(ttl)
    ax.grid(axis='y',color='#e6e6e6',zorder=0)
axs[0].axhline(0.002,color='#b3261e',ls='--',lw=1)
axs[0].text(0.2,0.00205,'0.002',color='#b3261e',fontsize=7.5,va='bottom')
axs[1].legend(handles=[Line2D([0],[0],marker='s',color='w',markerfacecolor=C[d],markersize=9,label=L[d])
                       for d in order],loc='upper right',frameon=False,fontsize=8)
fig.suptitle('Calibration of probabilities on the held-out source pool, per class',fontsize=10.5,y=1.02)
fig.tight_layout(); fig.savefig(RD/'calibration_reliability.png',bbox_inches='tight',facecolor='white')
print('Figure 6 regenerated with', len(order), 'environments and', len(cq), 'class bars')
print('per-dataset mean ECE:', cq.groupby('dataset')['ece'].mean().round(5).to_dict())


Figure 6 regenerated with 4 environments and 20 class bars
per-dataset mean ECE: {'cicids2017': 0.0001, 'ciciot2023': 0.00056, 'nslkdd': 0.0002, 'ugr16': 0.00118}


In [ ]:
# =============================================================================
# Cell 5 - refresh the ledger so the new numbers are verifiable, then commit.
# =============================================================================
led=RD/'final_results.json'
if led.exists():
    L=json.loads(led.read_text())
    L.setdefault('calibration',{})
    L['calibration']['mean_ece_by_dataset']={k:round(float(v),5)
        for k,v in cq.groupby('dataset')['ece'].mean().items()}
    L['calibration']['max_per_class_ece']=round(float(cq.ece.max()),5)
    L['calibration']['ciciot2023_toplabel_ece']=round(float(tl.toplabel_ece.mean()),5)
    L['_meta']['entries']['calibration.ciciot2023_toplabel_ece']={'source':'calibration_toplabel_ciciot2023.csv'}
    led.write_text(json.dumps(L,indent=2,default=str))
    print('ledger refreshed with the four-dataset calibration figures')
    print('  (re-run notebook 36 for a full regeneration and manuscript check)')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','step 3: CIC-IoT-2023 calibration quality; Figure 6 regenerated with all four environments')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


ledger refreshed with the four-dataset calibration figures
  (re-run notebook 36 for a full regeneration and manuscript check)
